# CalCourse Exploratory Data Analysis

This notebook explores the Fall 2026 Berkeley course and class datasets used in CalCourse. 

Goals are to: 
- understand the structure of the data
- identify missing or inconsistent fields 
- define the undergraduate recommendation pool
- inspect prerequisite and course metadata before modeling

## 1. Load Data

In [29]:
import pandas as pd

courses = pd.read_csv("../data/processed/courses_fall_2026.csv")
classes = pd.read_csv("../data/processed/classes_fall_2026.csv")

## 2. Dataset Overview
Inspecting dataset dimensions, schema, and sample record. 

In [30]:
print(courses.shape)
print(classes.shape)
print(courses.head())
print(classes.head())

(4287, 10)
(15573, 13)
   course_id  subject course_number  \
0     161725  AEROENG             1   
1     162181  AEROENG            10   
2     166420  AEROENG           100   
3     168794  AEROENG           122   
4     164194  AEROENG           197   

                                               title  \
0                    Aerospace Engineering 1 Seminar   
1       Introduction to Aerospace Engineering Design   
2                                 Aerospace Capstone   
3         Earth Observation Systems and Technologies   
4  Undergraduate Aerospace Engineering Field Studies   

                                         description  \
0  This is a freshman-level seminar course offere...   
1  This course introduces mathematical engineerin...   
2  This capstone course challenges students to in...   
3  This course introduces engineering students to...   
4  Supervised field experience relative to specif...   

                                        requirements academic_career

In [31]:
courses.info()
classes.info()

<class 'pandas.DataFrame'>
RangeIndex: 4287 entries, 0 to 4286
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   course_id             4287 non-null   int64  
 1   subject               4287 non-null   str    
 2   course_number         4287 non-null   str    
 3   title                 4287 non-null   str    
 4   description           4093 non-null   str    
 5   requirements          2326 non-null   str    
 6   academic_career       4287 non-null   str    
 7   department            4287 non-null   str    
 8   department_nicknames  0 non-null      float64
 9   typically_offered     0 non-null      float64
dtypes: float64(2), int64(1), str(7)
memory usage: 335.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 15573 entries, 0 to 15572
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   course_id      15573 non-null  int64  
 1

### Initial observations 
- The course dataset contains 4,287 unique courses. 
- The class dataset contains 15,573 Fall 2026 class records. 
- Course-level and class-level data are stored separately because one course may correspond to multiple class offerings.

## 3. Missing Data
Evaluating which fields are complete enough to use and which may require exclusion or special handling. 

In [32]:
courses.isna().sum().sort_values(ascending=False)
# classes.isna().sum().sort_values(ascending=False)

department_nicknames    4287
typically_offered       4287
requirements            1961
description              194
course_id                  0
subject                    0
course_number              0
title                      0
academic_career            0
department                 0
dtype: int64

In [33]:
courses.loc[
        courses["requirements"].notna(),
        ["subject", "course_number", "title", "academic_career"]
].head(20)

,subject,course_number,title,academic_career
1,AEROENG,10,Introduction to Aerospace Engineering Design,UGRD
2,AEROENG,100,Aerospace Capstone,UGRD
5,AEROENG,C124,Materials for Extreme Environments,UGRD
6,AEROENG,C136,Dynamics and Control of Autonomous Flight,UGRD
7,AEROENG,C136S,Software Laboratory: Dynamics and Control of A...,UGRD
8,MECENG,C162,Introduction to Flight Mechanics,UGRD
9,MECENG,C166,Introduction to Compressible Flow,UGRD
11,AEROSPC,100,Leadership Laboratory,UGRD
12,AEROSPC,135A,Leadership Studies,UGRD
15,AFRICAM,100,Black Intellectual Thought,UGRD


In [34]:
courses.loc[
    courses["description"].isna(),
    ["subject", "course_number", "title", "academic_career"]
].head(20)

,subject,course_number,title,academic_career
22,AFRICAM,136,Criminal Justice and the Community,UGRD
110,ANTHRO,230,Special Topics in Archaeology,GRAD
113,ANTHRO,250X,Seminars in Social and Cultural Anthropology: ...,GRAD
178,ARCH,249,Special Topics in the Physical Environment in ...,GRAD
185,ARCH,271,Methods in Historical Research and Criticism i...,GRAD
202,ARESEC,299,Individual Research,GRAD
285,ASTRON,199,Supervised Independent Study and Research,UGRD
292,ASTRON,299,Advanced Study and Research,GRAD
303,PHYSICS,C290C,Cosmology,GRAD
359,BUDDSTD,299,Thesis Preparation and Related Research,GRAD


### Missing data observations 
- 'department_nicknames' and 'typically_offered' are completely null and will likely be excluded. 
- 'requirements' is missing for many courses, but they may just indicate the absence of explicit prerequisites rather than a data-quality issue. 
- Course titles, identifiers, subjects, academic career, and department fields are complete.
- Missing descriptions are relatively uncommon and appear to be concentrated in certain research, seminar, or special-topic courses. 

## 4. Course Population 
Examining the size and composition of the course catalog to define the recommendation pool.

In [35]:
courses["academic_career"].value_counts()

academic_career
UGRD    2306
GRAD    1789
LAW      192
Name: count, dtype: int64

In [36]:
courses["subject"].nunique()

212

In [37]:
courses["department"].value_counts().head(20)

department
Haas School of Business          331
School of Law                    192
Political Science                170
Ethnic Studies                   135
Electrical Eng & Computer Sci    130
Env Sci, Policy, & Mgmt          111
School of Public Health           98
Music                             95
Mathematics                       94
Physics                           76
History                           74
East Asian Lang & Culture         72
UG Interdisciplinary Studies      71
Economics                         71
Integrative Biology               70
Molecular & Cell Biology          66
Psychology                        64
Sociology                         62
Mechanical Engineering            60
Interdisc Social Science Pgms     60
Name: count, dtype: int64

In [38]:
courses["course_id"].nunique(), len(courses)

(4287, 4287)

In [39]:
# CalCourse V1 will filter for undergrad

ug = courses[courses["academic_career"] == "UGRD"]
ug.shape

(2306, 10)

In [40]:
ug["course_number"].sample(50, random_state=42)

3101       99
851      C4AC
3546      166
52       C181
3120      198
1947     98BC
2522    C140V
3137     H194
3881      127
2887      199
3016     26AC
741      C100
2750       24
314       121
2984    168BS
848     C131A
3537      149
442     C191A
1411        2
788       110
2742      197
1854     100A
4051      169
1118     100A
3031       57
1861      15A
666       20B
4103      122
3104      C61
2539      130
2049      159
4107      128
3358      129
1392      172
1909     175E
3504     124M
1236     115B
29        165
2770     133L
1394      174
3959     98BC
3343       12
3350        4
1835      199
1293      143
2104     100C
4228     50AC
3071      12B
665       20A
4138     190D
Name: course_number, dtype: str

### Course Population Observations
- The dataset contains 4,287 unique Fall 2026 courses. 
- 2,306 courses are undergraduate and will form the initial CalCourse recommendation pool. 
- The catalog spans 212 subjects which provides broad coverage across departments. 
- Course numbers contain non-numeric prefixes and suffixes such as 'C', 'H', and 'AC', so they must be treated as strings rather than numeric values. 

## 5. Prerequisite Structure
Prereq information is stored as semi-structured text. This section looks at how prerequisites are written and how they can be handled in the recommendation system. 

In [41]:
# Prerequisite inspection

ug.loc[
    ug["requirements"].notna(),
    ["subject", "course_number", "title", "requirements"]
].sample(
    20, 
    random_state=42
)

,subject,course_number,title,requirements
3410,PHYSICS,89,Introduction to Mathematical Physics,Math 53; Physics 5A or 7A (can be taken concur...
3508,POLSCI,126A,International Political Economy,"College-level economics course (macro, micro, ..."
2213,KOREAN,100AX,Advanced Korean for Heritage Speakers,Korean 10BX; or consent of instructor.
2535,MATH,124,Programming for Mathematical Applications,"Math 53, 54, 55"
2492,LINGUIS,1A,American Sign Language I,Not open to native signers.
636,COLWRIT,143A,Academic Writing for Multilingual Students,Enrollment in College Writing 143A: ESP- Acade...
3843,SLAVIC,100L,"Advanced Readings in Russian, East European an...",Consent of instructor. Knowledge of an appropr...
2744,MBN,199,Supervised Independent Study and Research,Upper division standing and consent of instruc...
3973,SPANISH,2,Elementary Spanish - Second Semester,1 or equivalent.
208,ART,102,Advanced Painting: Research and Methods,ART 8 and ART 13 or equivalents.


## 6. Recommendation Pool
Not every undergrad course is useful as a general course rec. This section identifies courses such as independent study, thesis, research, and field-study offerings that may need to be handled separately.

In [42]:
keywords = [
    "independent study", 
    "independent research", 
    "directed study", 
    "field studies", 
    "thesis", 
    "supervised research"
]

pattern = "|".join(keywords)

special_courses = ug[
    ug["title"].str.contains(pattern, case=False, na=False)
]

special_courses[
    ["subject", "course_number", "title"]
].head(30)

,subject,course_number,title
4,AEROENG,197,Undergraduate Aerospace Engineering Field Studies
36,AFRICAM,199,Supervised Independent Study and Research
76,AMERSTD,190,Senior Thesis
77,AMERSTD,191,Senior Thesis Seminar
78,AMERSTD,199,Supervised Independent Study and Research for ...
81,AMERSTD,99,Supervised Independent Study and Research
88,AMERSTD,H195,Senior Honors Thesis Seminar
107,ANTHRO,199,Supervised Independent Study
127,ANTHRO,99,Supervised Independent Study and Research
154,ARCH,199,Supervised Independent Study and Research


In [43]:
special_courses.shape

(176, 10)

In [45]:
recommendable = ug[
    ~ug["title"].str.contains(pattern, case=False, na=False)
].copy()

recommendable.shape

(2130, 10)

In [46]:
## are there still courses in recommendable that probably shouldn't be default recommendations? 

recommendable["description"].isna().sum()

recommendable.loc[
    recommendable["description"].isna(),
    ["subject", "course_number", "title"]
].head(30)

,subject,course_number,title
22,AFRICAM,136,Criminal Justice and the Community
920,DUTCH,198,Directed Group Study
1724,GEOG,198,Directed Group Study
3206,PBHLTH,198,Directed Group Study
3324,PHILOS,160,Plato
3413,PHYSICS,98,Directed Group Study
3462,POLECON,198,Directed Group Study
3542,POLSCI,149W,Topics In Area Stdy
3793,RUSSIAN,2,Elementary Russian
3795,RUSSIAN,3,Intermediate Russian


In [47]:
# inspecting formats like seminar / deCals/ special topics

recommendable.loc[
    recommendable["title"].str.contains(
        "seminar|special topics|decal",
        case=False, 
        na=False
    ),
    ["subject", "course_number", "title"]
].head(40)

,subject,course_number,title
0,AEROENG,1,Aerospace Engineering 1 Seminar
27,AFRICAM,159,Special Topics in African American Literature
32,AFRICAM,194A,African American Theme Program Seminar
39,AFRICAM,24,Freshman Seminars
45,AFRICAM,40,Special Topics
60,AGRS,24,Freshman Seminars
73,AMERSTD,110,Special Topics in American Studies
79,AMERSTD,24,Freshman Seminar
87,AMERSTD,H110,Honors Seminar: Special Topics in American Stu...
91,ANTHRO,112,Special Topics in Biological Anthropology


In [48]:
# exclude titles that are clearly individualized or special-access

exclude_pattern = (
    "independent study|independent research|directed study|"
    "field studies|thesis|supervised research|"
    "research seminar|thesis seminar|honors seminar"
)

recommendable_v1 = ug[
    ~ug["title"].str.contains(
        exclude_pattern, 
        case=False, 
        na=False
    )
].copy()

recommendable_v1.shape

(2119, 10)

In [49]:
excluded_v1 = ug[
    ug["title"].str.contains(
        exclude_pattern,
        case=False,
        na=False
    )
]

excluded_v1[
    ["subject", "course_number", "title"]
].head(40)

,subject,course_number,title
4,AEROENG,197,Undergraduate Aerospace Engineering Field Studies
36,AFRICAM,199,Supervised Independent Study and Research
76,AMERSTD,190,Senior Thesis
77,AMERSTD,191,Senior Thesis Seminar
78,AMERSTD,199,Supervised Independent Study and Research for ...
81,AMERSTD,99,Supervised Independent Study and Research
87,AMERSTD,H110,Honors Seminar: Special Topics in American Stu...
88,AMERSTD,H195,Senior Honors Thesis Seminar
107,ANTHRO,199,Supervised Independent Study
127,ANTHRO,99,Supervised Independent Study and Research


### Recommendation pool observations
- CalCourse V1 starts with 2,306 undergrad courses.
- 187 courses are excluded because they're clearly individualized, research/thesis-oriented, field-study-based, or intended for special academic arrangements.
- Standard undergrad courses, including DeCals and regular seminars, remain eligible for recommendation. 
- After filtering, 2,119 courses remain in the recommendation pool.

## 7. Prerequisite Handling
Course requirements are stored as semi-structured text. This section looks at the most common prerequisite patterns anc indentifies which parts can be handled. 

In [50]:
prereq_sample = recommendable_v1.loc[
    recommendable_v1["requirements"].notna(), 
    ["subject", "course_number", "title", "requirements"]
].sample(
    30,
    random_state=42
)

prereq_sample

,subject,course_number,title,requirements
3653,PSYCH,130,Clinical Psychology,Recommended: Psychology 1 or Psychology 2
798,CYPLAN,199,Special Study for Advanced Undergraduates,Consent of instructor.
342,CMPBIO,C149,Computational Functional Genomics,MATH 54 or ELENG 64/ELENG 66; COMPSCI 61A or e...
943,ECON,136,Financial Economics,"100A or 101A, and one semester of statistics."
2915,MELC,R1A,Reading and Composition in Middle Eastern Lang...,Satisfaction of the Entry Level Writing Requir...
503,CHMENG,162,Dynamics and Control of Chemical Processes,Chemical and Biomolecular Engineering 142 and ...
3402,PHYSICS,5CL,Introduction to Experimental Physics II,Physics 5B & 5BL or 7B; Physics 5C or 7C (whic...
2035,INTEGBI,112,Horticultural Methods in the Botanical Garden,Consent of instructor.
3657,PSYCH,140,Developmental Psychology,Recommended: Psych 1
2149,JAPAN,155,Modern Japanese Literature,Japanese 100A (may be taken concurrently).


In [51]:
recommendable_v1["requirements"].notna().sum()

np.int64(1157)

In [52]:
recommendable_v1["requirements"].notna().mean()

np.float64(0.5460122699386503)

### Prerequisite observations
- 1,157 of 2,119 recommendable courses (54.6%) include a requirements field. 
- Requirements are semi-structured and mix explicit course prereqs with standing, consent, grade, audition, and equivalency constraints.
- CalCourse will extract recognizable course prereqs while retaining the original requirement text for constraints that cannot be reliably parsed. 

## 8. Final Recommendation Dataset
The final dataset keeps standard undergrad courses that are suitable for general recommendations and removes fields that are not usable in the current pipeline.

In [53]:
final_courses = recommendable_v1.drop(
    columns=[
        "department_nicknames", 
        "typically_offered"
    ]
).copy()

final_courses.shape

(2119, 8)

In [54]:
final_courses.isna().sum().sort_values(ascending=False)

requirements       962
description         11
course_id            0
subject              0
course_number        0
title                0
academic_career      0
department           0
dtype: int64

In [55]:
final_courses.sample(
    10, 
    random_state=42
)

,course_id,subject,course_number,title,description,requirements,academic_career,department
3841,122008,SEASIAN,R5B,Under Western Eyes,"In this course, the student will read selectio...",Previously passed an R_A course with a letter ...,UGRD,South & SE Asian Studies
696,104277,COMPSCI,164,Programming Languages and Compilers,Survey of programming languages. The design of...,COMPSCI 61B and COMPSCI 61C.,UGRD,Electrical Eng & Computer Sci
1443,107640,ESPM,39,Freshman/Sophomore Seminar,Freshman and sophomore seminars offer lower di...,Priority given to freshmen and sophomores.,UGRD,"Env Sci, Policy, & Mgmt"
1319,107274,EPS,131,Geochemistry,Chemical reactions in geological processes. Th...,"100A-100B, Chemistry 1A-1B.",UGRD,Earth & Planetary Science
3531,119061,POLSCI,143B,Japanese Politics,The structure and evolution of political insti...,NaN,UGRD,Political Science
1912,161893,HISTORY,188E,Eros: A History of Love from Ancient Greece to...,"What is love? An instinct, a thing of nature? ...",NaN,UGRD,History
3875,121132,SOCIOL,110,Organizations and Social Institutions,This survey course studies administrative orga...,"1, 3 or 3AC or consent of instructor.",UGRD,Sociology
3527,119005,POLSCI,141A,Russian Politics,This course investigates contemporary politics...,NaN,UGRD,Political Science
4239,122987,XMATH,3,Precalculus,"Polynomial and rational functions, exponential...","Three years of high school mathematics, plus s...",UGRD,UCB Extension
3036,161753,MUSIC,90,Making Music,"Introduction to creative music research, theor...",NaN,UGRD,Music


In [56]:
final_courses["course_id"].nunique(), len(final_courses)

(2119, 2119)

In [57]:
final_courses.to_csv(
    "../data/processed/recommendable_courses_fall_2026.csv",
    index=False
)

### Final dataset
- CalCourse contains 2,119 undergrad courses eligible for general recommendation.
- Fully null metadata fields were removed. 
- Missing prerequisite text is retained because it may indicate that a course has no explicit prerequisite.
- Original requirement text is preserved for later prerequisite extraction and eligibility handling. 

## 9. EDA Summary

The Fall 2026 Berkeley dataset was narrowed from 4,287 unique courses to a recommendation pool of 2,119 undergraduate courses.

Key findings: 
- Course-level and class-level data are stored separately, allowing one course to map to multiple class offerings. 
- Course identifiers and titles are complete, while some descriptive metadata is partially missing. 
- Course numbers are not purely numeric and must be treated as strings. 
- Prerequisite information is semi-structured and mixes explicit course requirements with standing, consent, grade, and equivalency constraints. 
- Courses intended for independent study, thesis work, research, or other individualized arrangements were excluded from the default recommendation pool.
- The final cleaned dataset preserves course descriptions and requirement text for downstream ranking and prerequisite processing. 

In [58]:
final_courses.shape

(2119, 8)

In [59]:
final_courses.head()

,course_id,subject,course_number,title,description,requirements,academic_career,department
0,161725,AEROENG,1,Aerospace Engineering 1 Seminar,This is a freshman-level seminar course offere...,NaN,UGRD,Mechanical Engineering
1,162181,AEROENG,10,Introduction to Aerospace Engineering Design,This course introduces mathematical engineerin...,"Prerequisite: MATH 51, MATH 52, MATH 53 (may b...",UGRD,Mechanical Engineering
2,166420,AEROENG,100,Aerospace Capstone,This capstone course challenges students to in...,MECENG 103; MECENG 104; MECENG 132; and MECENG...,UGRD,Mechanical Engineering
3,168794,AEROENG,122,Earth Observation Systems and Technologies,This course introduces engineering students to...,NaN,UGRD,Mechanical Engineering
5,168514,AEROENG,C124,Materials for Extreme Environments,This course introduces engineering students to...,ENGIN 40 (Thermodynamics of Materials) or MEC ...,UGRD,Mechanical Engineering
